# MBPS Pseudo-Label Pipeline Visualization
## k=80 + DepthPro + DCFA + SIMCF-ABC

This notebook visualizes each stage of the pseudo-label generation pipeline for 4 sample Cityscapes images.

**Pipeline Stages:**
1. **Input**: RGB (1024×2048) + DepthPro Depth (512×1024)
2. **Stage 1 (Semantic)**: DINOv2 ViT-B/14 → CAUSE 90D codes → DCFA adapter → k=80 MiniBatchKMeans → cluster-to-class mapping
3. **Stage 2 (Instance)**: DepthPro depth → Sobel edges → per-class connected components → area filter → dilation
4. **Stage 3 (SIMCF-ABC)**: Label-level consistency filtering (A: majority vote, B: instance merging, C: depth outlier masking)
5. **Stage 4 (Panoptic)**: Semantic + instance → Cityscapes panoptic encoding

**Note on Features:**
- Semantic generation uses **DINOv2 ViT-B/14** (CAUSE/Segment_TR), verified from `generate_depth_overclustered_semantics.py`
- Instance generation uses **DepthPro only**, no vision features, verified from `generate_depth_guided_instances.py`
- SIMCF Step B in `refine_simcf.py` contains code to load DINOv3 768-D features for cosine-similarity merging. Local feature files exist. Raw→refined instance count drops (15→8, 14→4) indicate Step B did run. The author states DINOv3 is not part of their intended pipeline design.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from PIL import Image
from scipy.ndimage import sobel, gaussian_filter
import torch
from collections import Counter

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# Paths
CS_ROOT = "/Users/qbit-glitch/Desktop/datasets/cityscapes"
PSEUDO_DIR = os.path.join(CS_ROOT, "cups_pseudo_labels_dcfa_simcf_abc")
RAW_DIR = os.path.join(CS_ROOT, "cups_pseudo_labels_adapter_V3_tau020")

# Cityscapes class definitions
CS_NAMES = {
    0: "road", 1: "sidewalk", 2: "building", 3: "wall", 4: "fence",
    5: "pole", 6: "traffic_light", 7: "traffic_sign", 8: "vegetation",
    9: "terrain", 10: "sky", 11: "person", 12: "rider", 13: "car",
    14: "truck", 15: "bus", 16: "train", 17: "motorcycle", 18: "bicycle",
}

CS_COLORS = {
    0: (128, 64, 128), 1: (244, 35, 232), 2: (70, 70, 70), 3: (102, 102, 156),
    4: (190, 153, 153), 5: (153, 153, 153), 6: (250, 170, 30), 7: (220, 220, 0),
    8: (107, 142, 35), 9: (152, 251, 152), 10: (70, 130, 180), 11: (220, 20, 60),
    12: (255, 0, 0), 13: (0, 0, 142), 14: (0, 0, 70), 15: (0, 60, 100),
    16: (0, 80, 100), 17: (0, 0, 230), 18: (119, 11, 32),
}

THING_IDS = set(range(11, 19))

def semantic_to_color(sem_map):
    """Convert trainID semantic map to RGB color image."""
    h, w = sem_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for tid, color in CS_COLORS.items():
        rgb[sem_map == tid] = color
    return rgb

def instance_to_color(inst_map, bg_color=(0, 0, 0)):
    """Color each instance with a unique bright color."""
    h, w = inst_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    rgb[:] = bg_color
    ids = np.unique(inst_map)
    ids = ids[ids > 0]
    np.random.seed(42)
    colors = np.random.randint(50, 255, size=(max(len(ids), 1) + 10, 3), dtype=np.uint8)
    for idx, iid in enumerate(ids):
        rgb[inst_map == iid] = colors[idx]
    return rgb

def load_image_data(stem):
    """Load all data for a single image stem."""
    city = stem.split('_')[0]
    base = stem.replace('_leftImg8bit', '')
    
    # RGB
    rgb = np.array(Image.open(os.path.join(CS_ROOT, 'leftImg8bit', 'train', city, f'{stem}.png')))
    
    # DepthPro depth
    depth = np.load(os.path.join(CS_ROOT, 'depth_depthpro', 'train', city, f'{base}.npy'))
    
    # Refined pseudo-labels (DCFA + SIMCF-ABC)
    sem_ref = np.array(Image.open(os.path.join(PSEUDO_DIR, f'{stem}_semantic.png')))
    inst_ref = np.array(Image.open(os.path.join(PSEUDO_DIR, f'{stem}_instance.png')))
    
    # Raw pseudo-labels (before SIMCF)
    if os.path.exists(os.path.join(RAW_DIR, f'{stem}_semantic.png')):
        sem_raw = np.array(Image.open(os.path.join(RAW_DIR, f'{stem}_semantic.png')))
        inst_raw = np.array(Image.open(os.path.join(RAW_DIR, f'{stem}_instance.png')))
    else:
        sem_raw = None
        inst_raw = None
    
    return {
        'rgb': rgb,
        'depth': depth,
        'sem_ref': sem_ref,
        'inst_ref': inst_ref,
        'sem_raw': sem_raw,
        'inst_raw': inst_raw,
        'stem': stem,
    }

# Select 4 sample images
SAMPLE_STEMS = [
    'aachen_000000_000019_leftImg8bit',
    'aachen_000001_000019_leftImg8bit',
    'aachen_000002_000019_leftImg8bit',
    'aachen_000003_000019_leftImg8bit',
]

samples = [load_image_data(s) for s in SAMPLE_STEMS]
print(f"Loaded {len(samples)} sample images")

## 1. INPUT: RGB + DepthPro Depth Maps

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 20))

for i, sample in enumerate(samples):
    # RGB
    axes[i, 0].imshow(sample['rgb'])
    axes[i, 0].set_title(f"{sample['stem']}\nRGB Image (1024×2048)")
    axes[i, 0].axis('off')
    
    # DepthPro depth
    im = axes[i, 1].imshow(sample['depth'], cmap='viridis', vmin=0, vmax=1)
    axes[i, 1].set_title(f"DepthPro Depth (512×1024)\nRange: [0, 1]")
    axes[i, 1].axis('off')
    plt.colorbar(im, ax=axes[i, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.suptitle('Stage 0: Pipeline Inputs', fontsize=16, y=1.01)
plt.show()

## 2. STAGE 1: Semantic Pseudo-Label Generation

**Pipeline:** DINOv2 ViT-B/14 → CAUSE Segment_TR (90D codes) → DCFA Adapter (40K params, depth-conditioned) → MiniBatchKMeans (k=80) → Cluster-to-Class Majority Vote → Semantic Map

The semantic pseudo-labels are stored as **cluster IDs (0–79)** before mapping, but we visualize them after cluster-to-class mapping to 19 Cityscapes trainIDs.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, sample in enumerate(samples):
    sem = sample['sem_ref']
    
    # Cluster ID histogram (Stage 1 output before mapping)
    unique_clusters = np.unique(sem)
    unique_clusters = unique_clusters[unique_clusters < 80]  # exclude 255 (ignore)
    
    # Visualize semantic map (after cluster-to-class mapping)
    axes[0, i].imshow(semantic_to_color(sem))
    axes[0, i].set_title(f"{sample['stem']}\nSemantic Pseudo-Label\n{len(unique_clusters)} clusters used")
    axes[0, i].axis('off')
    
    # Histogram of cluster IDs
    counts = Counter(sem[sem < 80].flatten())
    clusters = sorted(counts.keys())
    values = [counts[c] for c in clusters]
    axes[1, i].bar(clusters, values, color='steelblue', edgecolor='black', linewidth=0.5)
    axes[1, i].set_xlabel('Cluster ID (0-79)')
    axes[1, i].set_ylabel('Pixel Count')
    axes[1, i].set_title(f'k=80 Cluster Distribution\n(ignore=255 masked)')
    axes[1, i].set_xlim(-1, 80)

# Add legend for semantic colors
legend_patches = [mpatches.Patch(color=np.array(v)/255, label=k) for k, v in CS_COLORS.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=10, bbox_to_anchor=(0.5, -0.05), fontsize=9)

plt.tight_layout()
plt.suptitle('Stage 1: Semantic Pseudo-Label Generation (DCFA + k=80 Overclustering)', fontsize=16, y=1.02)
plt.show()

## 3. STAGE 2: Depth-Guided Instance Generation

**Pipeline:** DepthPro Depth → Gaussian Smooth (σ=0) → Sobel Gradient (Gx, Gy) → ||∇D|| > τ=0.20 → Per-Class Connected Components → Area Filter (≥1000 px) → Dilation (3 iterations)

**No vision features are used.** Instance boundaries come purely from depth discontinuities.

In [ ]:
def compute_depth_edges(depth, grad_threshold=0.20, blur_sigma=0.0):
    """Compute depth edges using Sobel gradient magnitude."""
    if blur_sigma > 0:
        depth_smooth = gaussian_filter(depth.astype(np.float64), sigma=blur_sigma)
    else:
        depth_smooth = depth.astype(np.float64)
    gx = sobel(depth_smooth, axis=1)
    gy = sobel(depth_smooth, axis=0)
    grad_mag = np.sqrt(gx**2 + gy**2)
    edges = grad_mag > grad_threshold
    return grad_mag, edges

fig, axes = plt.subplots(4, 4, figsize=(18, 20))

for i, sample in enumerate(samples):
    depth = sample['depth']
    grad_mag, edges = compute_depth_edges(depth, grad_threshold=0.20, blur_sigma=0.0)
    inst = sample['inst_ref']
    
    # Depth gradient magnitude
    im = axes[i, 0].imshow(grad_mag, cmap='hot')
    axes[i, 0].set_title(f"Sobel ||∇D||\nMax={grad_mag.max():.3f}")
    axes[i, 0].axis('off')
    plt.colorbar(im, ax=axes[i, 0], fraction=0.046, pad=0.04)
    
    # Depth edges (thresholded)
    axes[i, 1].imshow(edges, cmap='gray')
    axes[i, 1].set_title(f"Depth Edges\nτ=0.20 | {(edges.sum() / edges.size * 100):.1f}% pixels")
    axes[i, 1].axis('off')
    
    # Instance masks (colored)
    inst_colored = instance_to_color(inst)
    axes[i, 2].imshow(inst_colored)
    n_inst = len(np.unique(inst)) - 1  # exclude 0
    axes[i, 2].set_title(f"Instance Masks\n{n_inst} valid instances")
    axes[i, 2].axis('off')
    
    # Instance overlay on RGB
    rgb_resized = np.array(Image.fromarray(sample['rgb']).resize((inst.shape[1], inst.shape[0])))
    overlay = rgb_resized * 0.5 + inst_colored * 0.5
    overlay = overlay.astype(np.uint8)
    axes[i, 3].imshow(overlay)
    axes[i, 3].set_title(f"Instance Overlay on RGB")
    axes[i, 3].axis('off')

plt.tight_layout()
plt.suptitle('Stage 2: Depth-Guided Instance Generation (DepthPro ONLY, no vision features)', fontsize=16, y=1.01)
plt.show()

## 4. STAGE 3: SIMCF-ABC Refinement — Before vs After

Compare raw instance masks (before SIMCF) with refined instance masks (after SIMCF-ABC).

- **Step A**: Instance validates semantics (majority vote — no-op for CUPS-derived labels)
- **Step B**: Semantics validate instances (merge adjacent same-class instances)
- **Step C**: Depth validates semantics (3-sigma outlier masking)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 20))

for i, sample in enumerate(samples):
    inst_raw = sample['inst_raw']
    inst_ref = sample['inst_ref']
    sem_ref = sample['sem_ref']
    
    if inst_raw is None:
        for j in range(4):
            axes[i, j].text(0.5, 0.5, 'Raw data not available', ha='center', va='center')
            axes[i, j].axis('off')
        continue
    
    # Raw instances
    n_raw = len(np.unique(inst_raw)) - 1
    axes[i, 0].imshow(instance_to_color(inst_raw))
    axes[i, 0].set_title(f"BEFORE SIMCF\n{n_raw} instances")
    axes[i, 0].axis('off')
    
    # Refined instances
    n_ref = len(np.unique(inst_ref)) - 1
    axes[i, 1].imshow(instance_to_color(inst_ref))
    axes[i, 1].set_title(f"AFTER SIMCF-ABC\n{n_ref} instances\n({n_raw - n_ref} merged)")
    axes[i, 1].axis('off')
    
    # Instance area comparison
    def get_areas(inst_map):
        ids = np.unique(inst_map)
        ids = ids[ids > 0]
        return [int((inst_map == iid).sum()) for iid in ids]
    
    areas_raw = get_areas(inst_raw)
    areas_ref = get_areas(inst_ref)
    
    axes[i, 2].hist(areas_raw, bins=20, alpha=0.6, label=f'Raw (n={n_raw})', color='coral', edgecolor='black')
    axes[i, 2].hist(areas_ref, bins=20, alpha=0.6, label=f'Refined (n={n_ref})', color='steelblue', edgecolor='black')
    axes[i, 2].set_xlabel('Instance Area (pixels)')
    axes[i, 2].set_ylabel('Count')
    axes[i, 2].set_title('Instance Area Distribution')
    axes[i, 2].legend()
    axes[i, 2].set_yscale('log')
    
    # Ignore mask from Step C (pixels marked as 255)
    ignore_mask = sem_ref == 255
    ignore_pct = ignore_mask.sum() / ignore_mask.size * 100
    axes[i, 3].imshow(ignore_mask, cmap='Reds')
    axes[i, 3].set_title(f"Step C: Ignore Mask\n{ignore_pct:.2f}% pixels (|D−μ_c| > 3σ_c)")
    axes[i, 3].axis('off')

plt.tight_layout()
plt.suptitle('Stage 3: SIMCF-ABC Refinement (A: majority vote | B: instance merge | C: depth outlier mask)', fontsize=16, y=1.01)
plt.show()

## 5. STAGE 4: Panoptic Merge

**Encoding:** `panoptic_id = class_id × 1000 + instance_id`

- Stuff pixels: `instance_id = 0` (uniform color per class)
- Thing pixels: `instance_id = 1, 2, 3, ...` (unique color per instance)

Algorithm:
1. Place things first (sorted by score, assign to unassigned pixels)
2. Fill stuff (remaining unassigned pixels get stuff class)
3. Fallback CC (uncovered thing pixels → new connected-component instances)

In [ ]:
def generate_panoptic_visualization(semantic, instance, stuff_ids=set(range(11)), thing_ids=set(range(11, 19))):
    """Generate a visual panoptic map from semantic + instance labels."""
    h, w = semantic.shape
    panoptic_rgb = np.zeros((h, w, 3), dtype=np.uint8)
    assigned = np.zeros((h, w), dtype=bool)
    
    # Step 1: Place things (colored by instance)
    inst_ids = np.unique(instance)
    inst_ids = inst_ids[inst_ids > 0]
    np.random.seed(42)
    max_colors = max(len(inst_ids) + 100, 256)
    thing_colors = np.random.randint(50, 255, size=(max_colors, 3), dtype=np.uint8)
    
    for idx, iid in enumerate(inst_ids):
        mask = instance == iid
        sem_vals = semantic[mask]
        sem_vals = sem_vals[sem_vals < 19]
        if len(sem_vals) == 0:
            continue
        majority_cls = int(np.bincount(sem_vals, minlength=19).argmax())
        if majority_cls not in thing_ids:
            continue
        valid_mask = mask & ~assigned
        if valid_mask.sum() < 10:
            continue
        panoptic_rgb[valid_mask] = thing_colors[idx]
        assigned[valid_mask] = True
    
    # Step 2: Place stuff (uniform class color)
    for cls in sorted(stuff_ids):
        mask = (semantic == cls) & ~assigned
        if mask.sum() < 64:
            continue
        color = CS_COLORS.get(cls, (128, 128, 128))
        panoptic_rgb[mask] = color
        assigned[mask] = True
    
    # Step 3: Fallback CC for unassigned thing pixels
    from scipy import ndimage
    for cls in sorted(thing_ids):
        remaining = (semantic == cls) & ~assigned
        if remaining.sum() < 10:
            continue
        labeled, n_cc = ndimage.label(remaining)
        for cc_id in range(1, n_cc + 1):
            cc_mask = labeled == cc_id
            if cc_mask.sum() < 10:
                continue
            panoptic_rgb[cc_mask] = thing_colors[len(inst_ids) + cc_id]
            assigned[cc_mask] = True
    
    return panoptic_rgb, assigned
fig, axes = plt.subplots(4, 3, figsize=(16, 20))
for i, sample in enumerate(samples):
    sem = sample['sem_ref']
    inst = sample['inst_ref']
    rgb = sample['rgb']
    
    # RGB resized to pseudo-label resolution
    rgb_resized = np.array(Image.fromarray(rgb).resize((sem.shape[1], sem.shape[0])))
    
    # Panoptic visualization
    panoptic_rgb, assigned = generate_panoptic_visualization(sem, inst)
    
    # RGB
    axes[i, 0].imshow(rgb_resized)
    axes[i, 0].set_title(f"{sample['stem']}\nRGB (resized)")
    axes[i, 0].axis('off')
    
    # Semantic only
    axes[i, 1].imshow(semantic_to_color(sem))
    axes[i, 1].set_title("Semantic Only\n(stuff + things, no instances)")
    axes[i, 1].axis('off')
    
    # Panoptic
    axes[i, 2].imshow(panoptic_rgb)
    n_inst = len(np.unique(inst)) - 1
    n_assigned = assigned.sum() / assigned.size * 100
    axes[i, 2].set_title(f"Panoptic Pseudo-Label\n{n_inst} thing instances\n{n_assigned:.1f}% pixels assigned")
    axes[i, 2].axis('off')
plt.tight_layout()
plt.suptitle('Stage 4: Panoptic Merge (semantic + instance → panoptic_id = class×1000 + instance)', fontsize=16, y=1.01)
plt.show()



## 6. Step C Deep Dive: Depth Outlier Masking

Visualize which pixels were masked as ignore (255) by Step C's 3-sigma depth outlier detection.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 20))

for i, sample in enumerate(samples):
    depth = sample['depth']
    sem = sample['sem_ref']
    rgb = sample['rgb']
    
    # Resize depth to match semantic resolution if needed
    if depth.shape != sem.shape:
        depth_resized = np.array(Image.fromarray(depth).resize((sem.shape[1], sem.shape[0])))
    else:
        depth_resized = depth
    
    rgb_resized = np.array(Image.fromarray(rgb).resize((sem.shape[1], sem.shape[0])))
    
    # Per-class depth statistics
    class_depths = {}
    for cls in range(19):
        mask = sem == cls
        if mask.any():
            class_depths[cls] = depth_resized[mask]
    
    # Depth map with outlier mask overlay
    ignore_mask = sem == 255
    depth_display = np.copy(depth_resized)
    depth_display_rgb = plt.cm.viridis(depth_display)[:, :, :3]
    depth_display_rgb[ignore_mask] = [1.0, 0.0, 0.0]  # Red for ignored
    
    axes[i, 0].imshow(rgb_resized)
    axes[i, 0].set_title("RGB")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(depth_display_rgb)
    ignore_pct = ignore_mask.sum() / ignore_mask.size * 100
    axes[i, 1].set_title(f"Depth + Step C Ignore Mask\n{ignore_pct:.2f}% in red")
    axes[i, 1].axis('off')
    
    # Per-class depth distribution
    colors_list = [np.array(CS_COLORS[c])/255 for c in class_depths.keys()]
    bp = axes[i, 2].boxplot(
        [class_depths[c] for c in class_depths.keys()],
        labels=[CS_NAMES.get(c, str(c)) for c in class_depths.keys()],
        patch_artist=True,
        showfliers=False,
    )
    for patch, color in zip(bp['boxes'], colors_list):
        patch.set_facecolor(color)
    axes[i, 2].set_ylabel('Depth (normalized)')
    axes[i, 2].set_title('Per-Class Depth Distribution')
    axes[i, 2].tick_params(axis='x', rotation=45)
    
    # Histogram of ignored pixels by class
    ignore_counts = {}
    for cls in range(19):
        # Pixels that WOULD be class cls but are ignored
        # We don't have the pre-Step-C semantic, so approximate from depth stats
        pass
    
    # Instead show depth edge + ignore overlay
    _, edges = compute_depth_edges(depth_resized, grad_threshold=0.20)
    edge_ignore = np.zeros((*sem.shape, 3), dtype=np.uint8)
    edge_ignore[edges] = [255, 255, 0]  # Yellow for edges
    edge_ignore[ignore_mask] = [255, 0, 0]  # Red for ignored
    axes[i, 3].imshow(edge_ignore)
    axes[i, 3].set_title("Depth Edges (yellow)\n+ Ignore Mask (red)")
    axes[i, 3].axis('off')

plt.tight_layout()
plt.suptitle('Step C: Depth Validates Semantics (3-Sigma Outlier Masking)', fontsize=16, y=1.01)
plt.show()

## 7. Pipeline Statistics Summary

Aggregate statistics across the 4 sample images.

In [ ]:
stats = []
for sample in samples:
    sem = sample['sem_ref']
    inst = sample['inst_ref']
    depth = sample['depth']
    
    n_clusters = len(np.unique(sem[sem < 80]))
    n_instances = len(np.unique(inst)) - 1
    n_ignore = (sem == 255).sum()
    ignore_pct = n_ignore / sem.size * 100
    
    inst_areas = [(inst == iid).sum() for iid in np.unique(inst) if iid > 0]
    median_area = np.median(inst_areas) if inst_areas else 0
    max_area = max(inst_areas) if inst_areas else 0
    
    stats.append({
        'Image': sample['stem'],
        'Clusters Used': n_clusters,
        'Instances': n_instances,
        'Ignore Pixels': f"{n_ignore:,}",
        'Ignore %': f"{ignore_pct:.2f}%",
        'Median Inst Area': f"{int(median_area):,}",
        'Max Inst Area': f"{int(max_area):,}",
    })

import pandas as pd
df = pd.DataFrame(stats)
print("\n=== Pipeline Output Statistics ===")
print(df.to_string(index=False))

# Also show before/after if raw available
print("\n=== SIMCF-ABC Instance Merge Effect ===")
for sample in samples:
    if sample['inst_raw'] is not None:
        n_raw = len(np.unique(sample['inst_raw'])) - 1
        n_ref = len(np.unique(sample['inst_ref'])) - 1
        print(f"{sample['stem']}: {n_raw} → {n_ref} instances ({n_raw - n_ref} merged)")

## 8. Full Pipeline Grid (All Stages Side-by-Side)

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(24, 18))

for i, sample in enumerate(samples):
    rgb = sample['rgb']
    depth = sample['depth']
    sem = sample['sem_ref']
    inst = sample['inst_ref']
    rgb_resized = np.array(Image.fromarray(rgb).resize((sem.shape[1], sem.shape[0])))
    
    # 0. RGB
    axes[i, 0].imshow(rgb_resized)
    axes[i, 0].set_title("Input RGB")
    axes[i, 0].axis('off')
    
    # 1. Depth
    im = axes[i, 1].imshow(depth, cmap='viridis', vmin=0, vmax=1)
    axes[i, 1].set_title("DepthPro Depth")
    axes[i, 1].axis('off')
    
    # 2. Semantic
    axes[i, 2].imshow(semantic_to_color(sem))
    axes[i, 2].set_title(f"Stage 1: Semantic\n(k=80 clusters)")
    axes[i, 2].axis('off')
    
    # 3. Instance
    n_inst = len(np.unique(inst)) - 1
    axes[i, 3].imshow(instance_to_color(inst))
    axes[i, 3].set_title(f"Stage 2: Instances\n({n_inst} objects)")
    axes[i, 3].axis('off')
    
    # 4. Panoptic
    panoptic_rgb, _ = generate_panoptic_visualization(sem, inst)
    axes[i, 4].imshow(panoptic_rgb)
    axes[i, 4].set_title("Stage 4: Panoptic")
    axes[i, 4].axis('off')
    
    # 5. Ignore mask
    ignore = sem == 255
    axes[i, 5].imshow(ignore, cmap='Reds')
    axes[i, 5].set_title(f"Step C: Ignore\n({ignore.sum()/ignore.size*100:.2f}%)")
    axes[i, 5].axis('off')

plt.tight_layout()
plt.suptitle('Complete Pipeline: Input → Stage 1 (Semantic) → Stage 2 (Instance) → Stage 4 (Panoptic) + Step C (Ignore)', fontsize=18, y=1.01)
plt.show()

---
## Appendix: Code Verification Notes

### Verified from Source Code

**`mbps_pytorch/generate_depth_overclustered_semantics.py`**
- Line 55: `from models.dinov2vit import dinov2_vit_base_14` → uses **DINOv2**
- Line 292: loads `dinov2_vit_base_14.pth` from CAUSE checkpoint
- Extracts 90D `Segment_TR` codes via sliding window over DINOv2 features
- DCFA adapter (if provided) adjusts 90D codes conditioned on sinusoidal depth encoding
- k-means clustering on adjusted codes produces 80 cluster IDs

**`mbps_pytorch/generate_depth_guided_instances.py`**
- Function `depth_guided_instances()` takes only `semantic`, `depth`, and parameters
- Computes Sobel gradient magnitude on depth map: `||∇D|| = √(Gx² + Gy²)`
- Thresholds at `grad_threshold` (default 0.05, production uses 0.20)
- Per-class connected components on `cls_mask & ~depth_edges`
- Filters by `min_area` (default 100, production uses 1000)
- Reclaims boundary pixels via `ndimage.binary_dilation` (3 iterations)
- **No vision features are loaded or used**

**`scripts/refine_simcf.py`**
- **Step A** (`step_a`): Majority vote within each instance region. Structurally no-op for CUPS-derived labels (0 pixels changed).
- **Step B** (`step_b`): Function signature includes `features: np.ndarray` parameter described as "L2-normalized DINOv3 features". Main loop attempts to load features from `dinov3_features/train/{city}/{stem}.npy`. If found, computes per-instance mean feature vectors and merges adjacent same-class instances with cosine similarity > 0.85. Local feature files exist with shape (2048, 768). Raw→refined instance count drops (15→8, 14→4) indicate Step B executed. Author states DINOv3 is not part of their intended pipeline.
- **Step C** (`step_c`): Two-pass algorithm. Pass 1 computes per-class depth mean μ_c and std σ_c across all 2,975 training images. Pass 2 masks pixels as ignore (255) where `|D(p) − μ_c| > 3σ_c`.

**`mbps_pytorch/generate_panoptic_pseudolabels.py`**
- `generate_panoptic_map()` implements 3-step merge:
  1. Place things (sort by score, majority semantic class under mask, assign to unassigned pixels only)
  2. Place stuff (fill remaining gaps, instance_id = 0)
  3. Fallback CC (uncovered thing pixels → new connected-component instances with score=0.1)
- Encoding: `panoptic_id = class_id × 1000 + instance_id` (Cityscapes standard)